In [37]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
import numpy as np

from testdata import mvn_with_correlation
x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)

In [38]:
import numpy as np
from numba import njit
from numba.experimental import jitclass
from numba.types import int64, float64
from optikon import Propositionalization

### Utility ###
###############

@njit
def argsort_columns(x):
    n, p = x.shape
    out = np.empty((n, p), dtype=np.int64)
    for j in range(p):
        out[:, j] = np.argsort(x[:, j])
    return out

@njit
def max_weighted_support_greedy(x, y, max_depth=5):
    n, p = x.shape
    orders = argsort_columns(x)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    cum_support_count = 0
    non_separable = 0

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=np.int64) # cursor buffer for order updates
    
    best_sum = np.sum(y)
    num_cond = 0

    for k in range(1, max_depth+1):
        cum_support_count += support_count
        sum_y = np.sum(y[orders[:support_count, 0]])
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):
            sum_left, sum_right = 0, sum_y
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                y_i = y[orders[i, j]]
                sum_left += y_i
                sum_right -= y_i
                if x[orders[i, j], j]==x[orders[i+1, j], j]:
                    non_separable += 1
                    continue

                if sum_left > best_sum:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_sum = sum_left
                    improvement = True
                elif sum_right > best_sum:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_sum = sum_right
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = best_s*(x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised?
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1

    res = Propositionalization(v[:num_cond], t[:num_cond], s[:num_cond])
    return res, best_sum, {'cum_support_count': cum_support_count,
                           'non_separable': non_separable}

res, val, stats = max_weighted_support_greedy(x, y)
res.as_conj_str(), val

('x4 <= 0.494 & x1 <= 1.514 & x2 >= -0.140', 21.443390305350473)

In [39]:
@jitclass
class WeightedSupport:

    w: float64[:]
    value: float64
    value_removed: float64
    value_remaining: float64

    def __init__(self, w):
        self.w = w
        self.value = np.sum(w)
        self.value_removed = 0
        self.value_remaining = self.value

    def support(self, support):
        self.value = np.sum(self.w[support])

    def reset(self):
        self.value_removed = 0
        self.value_remaining = self.value

    def remove(self, i):
        yi = self.w[i]
        self.value_removed += yi
        self.value_remaining -= yi

ws = WeightedSupport(y)
ws.remove(0)
ws.remove(1)
ws.value_removed

-0.006374642197908592

In [56]:
@njit
def greedy_maximization(x, obj, max_depth=5):
    n, p = x.shape
    orders = argsort_columns(x)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    cum_support_count = 0
    non_separable = 0

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=np.int64) # cursor buffer for order updates
    
    best_value = obj.value
    num_cond = 0

    for k in range(1, max_depth+1):
        cum_support_count += support_count

        obj.support(orders[:support_count, 0])
        
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):

            obj.reset()
            
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                
                obj.remove(orders[i, j])
                
                if x[orders[i, j], j]==x[orders[i+1, j], j]:
                    non_separable += 1
                    continue

                if obj.value_removed > best_value:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_value = obj.value_removed
                    improvement = True
                elif obj.value_remaining > best_value:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_value = obj.value_remaining
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = best_s*(x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised?
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1

    res = Propositionalization(v[:num_cond], t[:num_cond], s[:num_cond])
    return res, best_value, {'cum_support_count': cum_support_count,
                           'non_separable': non_separable}

res, val, stats = greedy_maximization(x, WeightedSupport(y))
res.as_conj_str(), val, len(res.support_all(x))

('x4 <= 0.494 & x1 <= 1.514 & x2 >= -0.140', 21.443390305350473, 44)

In [69]:
%timeit max_weighted_support_greedy(x, y)

6.72 μs ± 50.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
%timeit greedy_maximization(x, WeightedSupport(y))

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Unknown attribute 'value' of type array(float64, 1d, C)

File "../../../../../../var/folders/zw/qxvhv2ms1rx684818_y1cvl40000gn/T/ipykernel_64037/2671827867.py", line 17:
<source missing, REPL/exec in use?>

During: typing of get attribute at /var/folders/zw/qxvhv2ms1rx684818_y1cvl40000gn/T/ipykernel_64037/2671827867.py (17)

File "../../../../../../var/folders/zw/qxvhv2ms1rx684818_y1cvl40000gn/T/ipykernel_64037/2671827867.py", line 17:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference

In [63]:
@jitclass
class RelativeWeightedSupport:

    w: float64[:]
    power: float64
    lam: float64
    
    sum_all: float64
    sum_removed: float64
    sum_remaining: float64
    
    count_all: int64
    count_removed: int64
    count_remaining: int64

    value: float64
    value_removed: float64
    value_remaining: float64

    def __init__(self, w, norm=2, lam=0):
        self.w = w
        self.power = 1/norm
        self.lam = lam
        self.sum_all = np.sum(w)
        self.count_all = len(w)
        self.value = self.sum_all / (self.count_all**self.power + self.lam)
        self.reset()

    def support(self, support):
        self.sum_all = np.sum(self.w[support])
        self.count_all = len(support)
        self.value = self.sum_all / (self.count_all**self.power + self.lam)

    def reset(self):
        self.sum_removed = 0
        self.sum_remaining = self.sum_all
        self.count_removed = 0
        self.count_remaining = self.count_all
        self.value_removed = 0
        self.value_remaining = self.sum_remaining / (self.count_remaining**self.power + self.lam)

    def remove(self, i):
        yi = self.w[i]
        self.sum_removed += yi
        self.sum_remaining -= yi
        self.count_removed += 1
        self.count_remaining -= 1
        self.value_removed = self.sum_removed / (self.count_removed**self.power + self.lam)
        self.value_remaining = self.sum_remaining / (self.count_remaining**self.power + self.lam)

rws = RelativeWeightedSupport(y, lam=0)
rws.remove(0)
rws.remove(1)
rws.remove(2)
rws.remove(3)
rws.value_removed

0.3694740626992066

In [68]:
res, val, stats = greedy_maximization(x, RelativeWeightedSupport(y, 2, lam=10))
res.as_conj_str(), val, len(res.support_all(x))

('x4 <= -0.383 & x2 >= -0.813 & x4 >= -2.007 & x1 <= 1.514 & x2 >= -0.022',
 1.446626284286675,
 22)

In [ ]:
@jitclass
class NormalisedWeightedSupport:

    w: float64[:]
    u: float64[:]
    power: float64
    
    sum_all: float64
    sum_removed: float64
    sum_remaining: float64
    
    count_all: int64
    count_removed: int64
    count_remaining: int64

    value: float64
    value_removed: float64
    value_remaining: float64

    def __init__(self, w, norm=2):
        self.w = w
        self.power = 1/norm
        self.sum_all = np.sum(w)
        self.count_all = len(w)
        self.value = self.sum_all / self.count_all**self.power
        self.reset()

    def support(self, support):
        self.sum_all = np.sum(self.w[support])
        self.count_all = len(support)
        self.value = self.sum_all / self.count_all**self.power

    def reset(self):
        self.sum_removed = 0
        self.sum_remaining = self.sum_all
        self.count_removed = 0
        self.count_remaining = self.count_all
        self.value_removed = 0
        self.value_remaining = self.sum_remaining / self.count_remaining**self.power

    def remove(self, i):
        yi = self.w[i]
        self.sum_removed += yi
        self.sum_remaining -= yi
        self.count_removed += 1
        self.count_remaining -= 1
        self.value_removed = self.sum_removed / self.count_removed**self.power
        self.value_remaining = self.sum_remaining / self.count_remaining**self.power